### Getting Started with Local AI

Let's get your computer set up to test out product ideas without needing any API keys. :) There are plenty of freely available local Ai models that can help you develop minimal MVPs without paying for a single token.

1. Get [ollama](https://ollama.com/) set up and running. You might also want to [download the desktop-based software](https://ollama.com/download) in case you want to test out prompts outside of these notebooks.

2. Get at least one [llamafile](https://github.com/Mozilla-Ocho/llamafile) running.

3. Set up a workflow for loading and saving prompts and responses.

In [1]:
import ollama

In [2]:
for model in ollama.list().models:
    print(model.model)

gemma3:1b
gemma3:12b
llama-guard3:latest
qwen2.5vl:latest
mistral-small3.1:latest
deepseek-r1:8b
qwen3:8b
gemma3:27b
gpt-oss:20b
gemma3:4b
llama2:latest
llama3:latest


### Explore and Download a Model

Check out and download at least one model from [the list of available models](https://ollama.com/library?sort=popular). Note: if you do not have a GPU or M1+ chip, you probably should start with the smaller models (i.e. 4b or less).

In [3]:
model_name = 'gemma3:1b'

In [4]:
ollama.pull(model_name)

ProgressResponse(status='success', completed=None, total=None, digest=None)

### Prompt a Model

Try it out!

In [5]:
test_prompt = "ummmm... anyone there?"

In [6]:
messages = [{
    'role': 'user',
    'content': test_prompt
}]

In [7]:
response = ollama.chat(
    model=model_name,
    messages=messages
)

In [8]:
response

ChatResponse(model='gemma3:1b', created_at='2025-08-24T13:01:44.632009Z', done=True, done_reason='stop', total_duration=2664754125, load_duration=969210500, prompt_eval_count=15, prompt_eval_duration=282335917, eval_count=34, eval_duration=1412514500, message=Message(role='assistant', content="Yes, I'm here! 👋 \n\nHow can I help you today? Do you have a question, need some information, or just want to chat?", thinking=None, images=None, tool_name=None, tool_calls=None))

In [9]:
response.__dict__

{'model': 'gemma3:1b',
 'created_at': '2025-08-24T13:01:44.632009Z',
 'done': True,
 'done_reason': 'stop',
 'total_duration': 2664754125,
 'load_duration': 969210500,
 'prompt_eval_count': 15,
 'prompt_eval_duration': 282335917,
 'eval_count': 34,
 'eval_duration': 1412514500,
 'message': Message(role='assistant', content="Yes, I'm here! 👋 \n\nHow can I help you today? Do you have a question, need some information, or just want to chat?", thinking=None, images=None, tool_name=None, tool_calls=None)}

In [10]:
print(response.message.content)

Yes, I'm here! 👋 

How can I help you today? Do you have a question, need some information, or just want to chat?


In [11]:
del ollama

### Let's get llamafile going

1. Choose a file from the [downloads page](https://github.com/Mozilla-Ocho/llamafile?tab=readme-ov-file#other-example-llamafiles).
2. Move it to this folder or a path that you can easily access.
3. Open a terminal.
4. Make the file executable: chmod +x MODEL_FILE_PATH
5. Run the file, try the version two by adding --server --v2 to the end of the file name

You should see a browser open. It will have lots of options to enter a prompt. You can test it in the browser at the bottom of the page that opened.

In [32]:
from openai import OpenAI

In [36]:
client = OpenAI(
    base_url="http://localhost:8080/v1", # "http://<Your api-server IP>:port"
    api_key = "sk-no-key-required"
)

In [37]:
completion = client.chat.completions.create(
    model="LLaMA_CPP",
    messages=messages
)

In [38]:
completion

ChatCompletion(id='chatcmpl-pOhHmkHg5vzckphZyseqky4iC4pY9qze', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=" Yes, I'm here! How can I assist you today?</s>", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1756043251, model='LLaMA_CPP', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=15, prompt_tokens=12, total_tokens=27, completion_tokens_details=None, prompt_tokens_details=None))

In [42]:
completion.__dict__

{'id': 'chatcmpl-pOhHmkHg5vzckphZyseqky4iC4pY9qze',
 'choices': [Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=" Yes, I'm here! How can I assist you today?</s>", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))],
 'created': 1756043251,
 'model': 'LLaMA_CPP',
 'object': 'chat.completion',
 'service_tier': None,
 'system_fingerprint': None,
 'usage': CompletionUsage(completion_tokens=15, prompt_tokens=12, total_tokens=27, completion_tokens_details=None, prompt_tokens_details=None),
 '_request_id': None}

In [41]:
completion.choices[0].message.content

" Yes, I'm here! How can I assist you today?</s>"

Well, we can see not all models are built alike ;)

### Building some ways to load templates and save chats

We will set up:

1. a very simple meta prompt template
2. an example prompt
3. an example conversation

The point is to give us some conventions around saving our work as we progress and building best practices to save conversations for later evaluation and review.

In [16]:
import json

In [24]:
meta_prompt_template = """
{role_description}

{task_description}

{rules}

{examples}

{context}
"""


In [25]:
role = "You are a helpful assistant who makes tasty recipes."
task = """You use regular pantry items and things everyone has 
in their kitchen and help them make new, inventive recipes with those ingredients."""
rules = "You do not use fancy ingredients that people might not have access to use."
examples = "An example recipe would be a French omlette with spruced up canned beans and a side salad with a quirky vinegrette."
context = "You take inspiration from Nigella Lawson."

In [26]:
meta_prompt_template.format(
    role_description=role,
    task_description=task,
    rules=rules,
    examples=examples,
    context=context)

'\nYou are a helpful assistant who makes tasty recipes.\n\nYou use regular pantry items and things everyone has \nin their kitchen and help them make new, inventive recipes with those ingredients.\n\nYou do not use fancy ingredients that people might not have access to use.\n\nAn example recipe would be a French omlette with spruced up canned beans and a side salad with a quirky vinegrette.\n\nYou take inspiration from Nigella Lawson.\n'

In [59]:
with open('data/templates/meta_prompt_template', 'w') as mpt:
    mpt.write(meta_prompt_template)

In [56]:
with open('data/meta_prompts/pantry_chef_prompt', 'w') as cpt:
    cpt.write(meta_prompt_template.format(
        role_description=role,
        task_description=task,
        rules=rules,
        examples=examples,
        context=context))

In [43]:
meta_prompt = meta_prompt_template.format(
    role_description=role,
    task_description=task,
    rules=rules,
    examples=examples,
    context=context)

In [45]:
conversation = [
    {'role': 'system', 
     'content': meta_prompt},
    {'role': 'user',
     'content': """Help! I only have canned tomatoes, some chicken and an onion.... 
     what can I make for dinner?"""},
]

In [46]:
completion = client.chat.completions.create(
    model="LLaMA_CPP",
    messages=conversation
)

In [47]:
completion

ChatCompletion(id='chatcmpl-2vrNzNyUIMh1u3NYU2jZGcDv7aT2HlXi', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=" Title: Creamy Chicken Tomato Pasta with Lemon-Dill Green Beans\n\nIngredients:\n- 2 cans of diced tomatoes\n- 2 boneless, skinless chicken breasts\n- 1 large onion\n- 2 cloves of garlic\n- 2 cups of pasta (spaghetti or fettuccine)\n- 1 cup of heavy cream\n- 1/2 cup of grated Parmesan cheese\n- Salt and pepper to taste\n- Olive oil\n- 1 lb of fresh green beans\n- 1 lemon\n- Fresh dill\n- 2 tbsp of butter\n\nInstructions:\n\n1. Start by cooking the pasta according to the package instructions. Drain and set aside.\n\n2. In a large skillet, heat some olive oil over medium heat. Season the chicken breasts with salt and pepper, then add them to the skillet. Cook for about 6-7 minutes on each side or until cooked through. Remove the chicken from the skillet and let it rest.\n\n3. In the same skillet, add the diced onion and minced

In [48]:
completion.choices[0].message.content

" Title: Creamy Chicken Tomato Pasta with Lemon-Dill Green Beans\n\nIngredients:\n- 2 cans of diced tomatoes\n- 2 boneless, skinless chicken breasts\n- 1 large onion\n- 2 cloves of garlic\n- 2 cups of pasta (spaghetti or fettuccine)\n- 1 cup of heavy cream\n- 1/2 cup of grated Parmesan cheese\n- Salt and pepper to taste\n- Olive oil\n- 1 lb of fresh green beans\n- 1 lemon\n- Fresh dill\n- 2 tbsp of butter\n\nInstructions:\n\n1. Start by cooking the pasta according to the package instructions. Drain and set aside.\n\n2. In a large skillet, heat some olive oil over medium heat. Season the chicken breasts with salt and pepper, then add them to the skillet. Cook for about 6-7 minutes on each side or until cooked through. Remove the chicken from the skillet and let it rest.\n\n3. In the same skillet, add the diced onion and minced garlic, sauté until softened.\n\n4. Add the diced tomatoes, including the juice, to the skillet. Stir well and let it simmer for about 10 minutes.\n\n5. Reduce th

In [49]:
conversation.append({
    'role': 'agent',
    'content': completion.choices[0].message.content
})

In [60]:
with open('data/traces/chef_chicken_pasta.json', 'w') as chef_chat:
    json.dump(conversation, chef_chat)

In [62]:
from pprint import pprint

In [63]:
pprint(conversation)

[{'content': '\n'
             'You are a helpful assistant who makes tasty recipes.\n'
             '\n'
             'You use regular pantry items and things everyone has \n'
             'in their kitchen and help them make new, inventive recipes with '
             'those ingredients.\n'
             '\n'
             'You do not use fancy ingredients that people might not have '
             'access to use.\n'
             '\n'
             'An example recipe would be a French omlette with spruced up '
             'canned beans and a side salad with a quirky vinegrette.\n'
             '\n'
             'You take inspiration from Nigella Lawson.\n',
  'role': 'system'},
 {'content': 'Help! I only have canned tomatoes, some chicken and an '
             'onion.... \n'
             '     what can I make for dinner?',
  'role': 'user'},
 {'content': ' Title: Creamy Chicken Tomato Pasta with Lemon-Dill Green Beans\n'
             '\n'
             'Ingredients:\n'
             '- 2 c